# 05 — Out-of-distribution: coronavirus genomes

The RBP models, applied without retraining to eight human pathogenic coronavirus
genomes tiled into windows. No labels exist for the viral windows, so the
evaluation rests entirely on the uncertainty scores — the label-free setting this
framework is designed for (Fig. 11). The figure pools the five seeds (`VIRUS_sdshift_pooled`).

In [ ]:
import os, sys, pathlib

# repo root, whether the notebook is run from notebooks/ or from the root
ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

# released data: the summary bundle (all notebooks) and the raw per-pass predictions
# (notebooks 04 and 06). Set the variables before starting Jupyter to use other locations.
os.environ.setdefault('UQFIG_BUNDLE', str(ROOT / 'figure_data_bundle'))
os.environ.setdefault('UQFIG_PREDS', str(ROOT / 'cluster_preds'))
OUT = ROOT / 'out'; OUT.mkdir(exist_ok=True)

from uq_genomics.figures import style as S
S.apply_rc()
print('bundle :', os.environ['UQFIG_BUNDLE'])
print('preds  :', os.environ['UQFIG_PREDS'])
print('output :', OUT)

In [ ]:
from uq_genomics.figures import plots_sdshift as SDS
SDS.build_virus(str(OUT), seed='all')

In [ ]:
from IPython.display import IFrame, display
p = OUT / 'VIRUS_sdshift_pooled.pdf'
print(p, '->', 'exists' if p.exists() else 'MISSING')
display(IFrame(str(p), width=900, height=560)) if p.exists() else None

## Quantifying the shift (Table S9)

Pooled over five seeds and all eight genomes: the KS statistic between human and
viral score distributions, and — the practical framing — the fraction of viral
windows above the 95th percentile of the human scores, i.e. what a practitioner
without viral labels would flag. AUROC and TPR at 5% FPR are produced by
`scripts/ood/ood_metrics.py`.

In [ ]:
import numpy as np
from scipy.stats import ks_2samp
from uq_genomics.figures import config_paths as CP, plots_virus as PV
# Table S9 convention: per seed, all eight genomes pooled; then mean (min-max) over seeds.
rows = {m: {'ks': [], 'frac': []} for m in CP.MODELS}
for seed in CP.VERSIONS['rbp']['seeds']:
    got, err = PV._collect(seed)
    if not got: continue
    for m in CP.MODELS:
        sd_in, mean_in, per_v = got[m]
        sd_in = np.asarray(sd_in); mean_in = np.asarray(mean_in)
        human = sd_in.max(axis=1)                       # max SD over classes, as for the viral windows
        viral = np.concatenate([np.asarray(a).max(axis=1) for a in per_v.values()])
        thr = np.percentile(human, 95)
        rows[m]['ks'].append(ks_2samp(human, viral).statistic)
        rows[m]['frac'].append((viral > thr).mean())
for m in CP.MODELS:
    ks, fr = np.array(rows[m]['ks']), np.array(rows[m]['frac'])
    print(f"{CP.MODEL_LABEL[m]:4s} KS {ks.mean():.2f} ({ks.min():.2f}-{ks.max():.2f})   "
          f"flagged {100*fr.mean():.0f}% ({100*fr.min():.0f}-{100*fr.max():.0f})")
print("AUROC and TPR at 5% FPR: scripts/ood/ood_metrics.py")